# Lakeflow Jobs — Orchestration, Triggers, Control Flow

## What's covered

- **Lakeflow Jobs** — what a *job* is, what a *task* is, what a *run* is
- **The DAG** — declaring dependencies between tasks; multi-task vs. single-task jobs
- **Task types** the exam tests by name:
  - **Notebook** task
  - **SQL** task — query, alert, file, dashboard
  - **Pipeline** task — runs a Lakeflow Spark Declarative Pipeline
  - **Dashboard** task — refreshes an AI/BI dashboard
  - Plus: JAR, Python script, Python wheel, dbt, run-job (nested job), `for_each`, `if/else`
- **Control flow** — `if/else` task branching, `for_each` looping, **retries** (job-level and task-level), task-level timeouts
- **Trigger types** — **scheduled** (cron), **file arrival**, **table update**, **continuous**, manual
- **Time-based vs. data-driven triggers** — and which scenario calls for which
- **Compute** — job cluster vs. all-purpose vs. serverless; per-task compute mix
- **Parameters, task values, and notifications**
- **Repair, run history, and rerun** — the operational view

## Reference domain

The bank's nightly **`fintech_nightly_ingest`** job is the worked example. It ingests daily drops for cards / accounts / loans, runs silver transformations, refreshes `customer_360`, fires the BI dashboard, and emails ops on failure.

## What is a Lakeflow Job?

Three nested concepts — get the vocabulary right and the rest is easy.

- **Job** — the unit of orchestration. A named, version-controlled definition of *what to run and when*. The bank has one nightly job per business unit.
- **Task** — one step in a job. Notebook, SQL, pipeline, dashboard, etc. A job is one task or many tasks in a DAG.
- **Run** — a single execution. "The job had 50 runs last month; 49 succeeded." A run has a state (pending / running / succeeded / failed / cancelled / skipped).

**Rename watch.** *Lakeflow Jobs* is the current name. *Databricks Workflows* and *Databricks Jobs* are the legacy names — the exam can use either.

**Single-task vs. multi-task.** A one-task job is fine for trivial work. The exam mostly tests **multi-task jobs** — a DAG of tasks with dependencies — because that's what real ETL looks like.

## The DAG — tasks and dependencies

Tasks declare which other tasks must succeed first. Lakeflow Jobs builds a directed acyclic graph from those dependencies and runs tasks in topological order, parallelising where it can.

The bank's nightly job:

```text
                  ┌── ingest_cards ───┐
                  │                   │
  start_marker ───┼── ingest_accounts ┼──► silver_build ──► gold_customer_360 ──► refresh_dashboard ──► notify
                  │                   │
                  └── ingest_loans ───┘
```

- `start_marker` is an optional gating task (often a SQL `SELECT 1` to confirm warehouses are up).
- The three `ingest_*` tasks run in parallel.
- `silver_build` declares all three as upstream — it runs once all three succeed.
- `refresh_dashboard` runs after `gold_customer_360` succeeds.
- `notify` runs unconditionally on completion (covered in *Run if* below).

**Three useful task settings the exam tests:**

- **`depends_on`** — list of upstream task keys. Empty list = a root task.
- **Run if** — when this task should run *given* upstream outcomes. Options include `ALL_SUCCESS` (default), `AT_LEAST_ONE_SUCCESS`, `NONE_FAILED`, `ALL_DONE`, `AT_LEAST_ONE_FAILED`, `ALL_FAILED`. *`ALL_DONE` is how you wire a final notification task that always fires.*
- **`max_retries`** — how many times to re-attempt the task on failure, plus an exponential backoff between attempts.

## Task types — the four the exam tests by name

Lakeflow Jobs supports a long list of task types. The four the exam asks about directly:

| Task type | What it runs | Where it goes |
|---|---|---|
| **Notebook** | A `.ipynb` notebook from Workspace or Git Folder | Cluster (job, all-purpose, or serverless) |
| **SQL** | A saved SQL query, alert, file, or dashboard | SQL warehouse |
| **Pipeline** | A Lakeflow Spark Declarative Pipeline (formerly DLT) | Pipeline compute (managed by the pipeline) |
| **Dashboard** | Refresh an AI/BI dashboard | SQL warehouse |

And the family of secondary task types you should recognise:

- **JAR**, **Python script**, **Python wheel** — code packaged outside notebooks (notebook 07 covers the bundle workflow).
- **dbt** — run a dbt project task.
- **Run job** — run another Lakeflow Job (nested job composition).
- **`for_each`** — loop a child task over an iterable of inputs (covered below).
- **`if/else` condition** — branching (covered below).

**The SQL task has four sub-variants** because DBSQL has four artifact types: queries, alerts, files, dashboards. A *SQL task — query* is the most common.

## Control flow — `if/else`, `for_each`, retries

Three control-flow primitives the exam can plant a question on.

**`if/else` condition task.** A task whose only job is to evaluate a boolean expression and route the DAG. Downstream tasks declare "only run if my parent's branch was `true`":

```text
                          ┌── true ──► run_fraud_rebuild
  is_friday  ──► if/else ─┤
                          └── false ─► skip_to_dashboard
```

**`for_each` task.** Runs a child task once per item in an array. Used to fan out the same notebook over multiple inputs:

```python
 # Parent for_each: iterate over the four verticals
 # inputs = ["cards", "accounts", "loans", "payments"]
 # Child notebook ingest.py runs four times, once per vertical, in parallel
```

The child task receives one item per execution as a parameter (`{{input}}` in templates).

**Retries.** Configurable at the task level:

- **`max_retries`** — how many times Databricks re-runs the task on failure.
- **`min_retry_interval_millis`** — minimum gap between attempts.
- **`retry_on_timeout`** — also retry when the task times out, not just on errors.
- **`timeout_seconds`** — kill the task if it runs longer than this.

**The exam pattern:** a flaky upstream API. Configure `max_retries = 3` on the ingest task. The first transient failure auto-retries; only persistent failures page humans.

## Trigger types — what *fires* a job run

Five trigger options. The first three are the ones the exam tests directly.

**Scheduled (cron / time-based).** Run at a specific time on a quartz cron expression.

```text
  cron: 0 0 2 * * ?         # every day at 02:00 UTC
  cron: 0 0 */4 * * ?       # every 4 hours
  cron: 0 0 9 ? * MON-FRI   # weekdays at 09:00 UTC
```

Best when the source side has a predictable cadence — the SFTP drops a file every night at 01:30, you schedule the job for 02:00.

**File arrival.** Run when new files land in a specific path. Lakeflow Jobs polls the path; when it sees new files, the job fires.

Best when the source side is irregular — the vendor drops a file *some time* after their batch ends; you don't want to schedule defensively at 04:00 only to find the file landed at 03:55 and a separate file landed at 04:30.

**Table update.** Run when a *Delta table* gets a new commit. Watch for upstream silver to update, then trigger the gold-rebuild job.

Best when the dependency is *table state*, not files or wall-clock time — your job consumes `silver.card_transactions`, you want gold to rebuild whenever silver changes.

**Continuous trigger.** Run forever, the streaming-pipeline pattern. Only relevant when the underlying task is a Lakeflow Spark Declarative Pipeline in continuous mode.

**Manual.** No trigger; only fires when a human or API call starts it. Useful for ad-hoc backfills.

## Time-based vs. data-driven triggers — the exam decision

The exam frames most trigger questions as a single tradeoff: *time-based* (scheduled cron) vs. *data-driven* (file arrival or table update).

**Time-based** when:

- The data lands on a predictable, well-known cadence.
- You want a stable execution window for SLAs and cost forecasting.
- Downstream consumers expect the table to be "fresh as of midnight" regardless of when the source actually landed.

**Data-driven** when:

- Source landing time is variable or unpredictable.
- You want to *minimise latency* — run as soon as data is available.
- Wasted runs (no data, just polled) would be expensive.

**The classic exam scenario:** a partner drops a file at an unpredictable time each night. The team wants to ingest as soon as the file is available, without running an empty job every hour. **File arrival** trigger is the answer. Cron-based is a distractor that wastes compute.

## Compute for jobs — per-task choice

Tasks within a job can use **different compute**. Three real choices:

- **Job cluster** — created at task start, torn down at task end. Cheapest DBU rate. Default for production ETL. Notebook 01 covered this.
- **Serverless compute (jobs)** — second-scale startup, no cluster management. Best for short tasks where 3-minute classic startup is wasted overhead.
- **All-purpose cluster** — long-lived, shared. **Avoid for scheduled jobs** — higher DBU rate and risk of cross-job interference.
- **SQL warehouse** — required for SQL and dashboard tasks. Serverless SQL warehouse is the modern default.

**Compute mix in the bank's nightly job:**

- `ingest_cards` (Auto Loader streaming notebook) — job cluster, DBR 14.x LTS
- `silver_build` (heavy SQL transformation notebook) — same job cluster reused (cluster reuse across tasks)
- `refresh_dashboard` (Dashboard task) — serverless SQL warehouse
- `notify` (Notebook task, tiny) — serverless jobs compute

**Cluster reuse across tasks** — set on the cluster definition. Several notebook tasks share one cluster, paying startup only once. Saves money for a chain of small steps.

## Parameters, task values, and task references

Tasks need to share state with each other. Three mechanisms.

**Job parameters.** Defined at the job level; visible to every task. The bank's nightly job declares a `processing_date` parameter; every notebook reads it.

**Task values.** A task can *set* a key-value pair that downstream tasks read.

```python
 # In an upstream notebook task:
 dbutils.jobs.taskValues.set(key="new_file_count", value=42)

 # In a downstream notebook task:
 count = dbutils.jobs.taskValues.get(taskKey="ingest_cards", key="new_file_count")
```

**Built-in references** — usable in any task parameter:

- `{{job.id}}`, `{{job.name}}`, `{{job.run_id}}`
- `{{job.start_time.iso_date}}` — the run start date
- `{{tasks.<task_key>.values.<key>}}` — task values from upstream
- `{{job.parameters.<param>}}` — job-level parameters

These are the right answer when a question asks how to pass a date or a row count between tasks without writing to a side table.

## Notifications and operational hooks

Three channels.

**Email notifications** — on start, success, failure, duration warning, streaming backlog. Configured at the job level.

**Webhooks / system destinations** — Slack, Microsoft Teams, PagerDuty, generic HTTP. The right answer when the question mentions paging on-call.

**Run-as identity** — a job runs *as* a user or service principal. The identity governs which UC permissions apply. Production jobs should run as a **service principal**, not a human's account.

**Always-run-on-finish task.** Wire a final task with `run_if = ALL_DONE` — it fires whether upstream succeeded or failed. The bank uses this to push a summary status to Slack after every nightly run.

## Monitoring, repair, rerun

The Lakeflow Jobs UI is the exam's go-to for monitoring questions. Three things to know:

**Run history view.** Lists every run of a job, with state, duration, and a sparkline of durations over time. Used to spot *trends* — "the job is taking 2× longer than three months ago."

**Task graph view.** Shows the DAG for a single run, coloured by state. Red = failed; grey = skipped; green = succeeded. Used to spot **upstream blockers** — if ingest_cards is red, silver_build is grey, and gold is grey, the root cause is ingest_cards.

**Repair run.** Rerun *only the failed tasks* of a previous run, with the same parameters, in the same DAG. Cheaper and safer than re-running the whole job.

**Run dashboard / Jobs metrics.** Aggregate views — success rate, p95 duration — for capacity and SLA tracking.

**The exam's monitoring question pattern:** a job's median duration tripled overnight. *Where do you look first?* → run history view for the trend, then task graph view of a slow run for the offending task. Notebook 08 goes deeper into Spark-level diagnostics from there.

## Defining a job — UI, REST API, and YAML

Three ways to define a job. The exam doesn't ask for syntax, but it does expect you to recognise the YAML shape, because that's how Automation Bundles (notebook 07) deploy them.

**UI** — point-and-click, fine for one-offs.

**Jobs REST API** — `POST /api/2.1/jobs/create` with a JSON body. The same JSON Automation Bundles emit.

**YAML in an Automation Bundle** — the production pattern. Here's a tiny end-to-end shape (full bundle structure is in notebook 07):

```yaml
 resources:
   jobs:
     fintech_nightly_ingest:
       name: fintech_nightly_ingest
       schedule:
         quartz_cron_expression: "0 0 2 * * ?"
         timezone_id: "UTC"
       email_notifications:
         on_failure: ["data-platform-oncall@bank.example"]
       tasks:
         - task_key: ingest_cards
           notebook_task:
             notebook_path: ../notebooks/ingest_cards.ipynb
           job_cluster_key: shared_etl_cluster
           max_retries: 3
         - task_key: silver_build
           depends_on: [{task_key: ingest_cards}]
           notebook_task: { notebook_path: ../notebooks/silver_build.ipynb }
           job_cluster_key: shared_etl_cluster
         - task_key: notify
           depends_on: [{task_key: silver_build}]
           run_if: ALL_DONE
           notebook_task: { notebook_path: ../notebooks/notify.ipynb }
       job_clusters:
         - job_cluster_key: shared_etl_cluster
           new_cluster:
             spark_version: "14.3.x-scala2.12"
             node_type_id: "i3.xlarge"
             num_workers: 4
```

Look at the structure, not the syntax. The DAG (`depends_on`), the trigger (`schedule`), the retries (`max_retries`), and the `run_if = ALL_DONE` notification pattern all match what the UI shows you.

## Hands-on — tiny utilities you'd write inside job tasks

Jobs themselves are defined in the workspace UI or via bundles. The cells below show the *notebook code* you'd write **inside** the tasks of the bank's nightly job — the bits that read parameters, set task values, decide branches, and emit summary metrics.

Run in a Databricks notebook attached to a cluster (DBR 14.x+). Outside a job, `dbutils.widgets.get` returns the default; inside a job, it returns the value the orchestrator passed.

In [ ]:
# 1) Inside a task notebook — pull job parameters via widgets.
#    The job definition passes {{job.parameters.processing_date}} as 'processing_date'.
dbutils.widgets.text("processing_date", "2026-06-18")
processing_date = dbutils.widgets.get("processing_date")
print(f"Running for processing_date = {processing_date}")

In [ ]:
# 2) Inside the ingest_cards task — count how many new files we just landed,
#    then set a task value so downstream tasks can read it.
new_file_count = 124   # in real life: dbutils.fs.ls(...) then len()
dbutils.jobs.taskValues.set(key="new_file_count", value=new_file_count)
print(f"Task value set: new_file_count = {new_file_count}")

In [ ]:
# 3) Inside a downstream task — read the upstream's task value.
#    Skipped if running outside a job context.
try:
    upstream_count = dbutils.jobs.taskValues.get(taskKey="ingest_cards", key="new_file_count", default=0)
    print(f"Downstream sees: upstream_count = {upstream_count}")
except Exception as e:
    print("Not running inside a Lakeflow Job — task values unavailable. Recipe shown.")

In [ ]:
# 4) Inside an if/else condition task you could express the predicate as a SQL expression.
#    In a notebook, the equivalent is: compute a boolean, set it as a task value,
#    and let the if/else condition task reference {{tasks.decide.values.should_rebuild_fraud}}.
from datetime import date
is_friday = date.today().weekday() == 4
dbutils.jobs.taskValues.set(key="should_rebuild_fraud", value=is_friday)
print(f"should_rebuild_fraud = {is_friday}")

In [ ]:
# 5) Final notify task (run_if = ALL_DONE): collect run metadata and print a summary line
#    a Slack webhook task could relay.
from datetime import datetime
summary = {
    "job":              "fintech_nightly_ingest",
    "processing_date":  processing_date,
    "new_file_count":   new_file_count,
    "finished_at":      datetime.utcnow().isoformat() + "Z",
}
print("SLACK SUMMARY:", summary)

In [ ]:
# Trigger-type recipes — what you'd put in the job definition (not runnable code,
# but the shapes the exam expects you to recognise).
print("""
## Scheduled (cron, time-based)
schedule:
  quartz_cron_expression: "0 0 2 * * ?"   # 02:00 UTC daily
  timezone_id: "UTC"

## File arrival (data-driven)
trigger:
  file_arrival:
    url: "/Volumes/fintech_dev/bronze/landing_zone/cards/"
    min_time_between_triggers_seconds: 60
    wait_after_last_change_seconds: 30

## Table update (data-driven)
trigger:
  table_update:
    table_names:
      - fintech_dev.silver.card_transactions
    condition: ANY_UPDATED

## Continuous (only valid when the underlying task is a Lakeflow Spark Declarative Pipeline)
continuous:
  pause_status: UNPAUSED
""")

## Section 4 self-check

**1.** A partner SFTPs a file at an unpredictable time each night between 02:00 and 06:00. The bank wants to ingest it as soon as it lands, without running empty hourly jobs. Which trigger fits?

- A. Scheduled cron at 06:30
- B. File arrival trigger on the landing volume
- C. Continuous trigger
- D. Manual trigger

**2.** A multi-task job has `ingest_cards`, `ingest_loans`, `silver_build`, and `notify`. The team wants `notify` to fire whether the upstream succeeded or failed. Which setting on `notify` makes that happen?

- A. `run_if = ALL_SUCCESS`
- B. `run_if = ALL_DONE`
- C. `run_if = AT_LEAST_ONE_SUCCESS`
- D. `max_retries = 0`

**3.** Which task type is correct for refreshing an AI/BI dashboard from a job?

- A. Notebook task
- B. SQL task — query
- C. Dashboard task
- D. Pipeline task

**4.** An ingest task fails about 1% of the time because the source API is flaky. The team wants automatic retry with exponential backoff before paging anyone. Which setting?

- A. `max_retries = 3` on the task
- B. `run_if = ALL_DONE`
- C. `for_each` over [1, 2, 3]
- D. Use a continuous trigger

**5.** A job's nightly DAG should rebuild gold whenever `silver.card_transactions` gets a new commit, no matter what time it lands. Which trigger fits?

- A. Scheduled cron
- B. File arrival
- C. Table update
- D. Manual

<details><summary>Show answers</summary>

1. **B** — file arrival fires only when files actually land.
2. **B** — `ALL_DONE` always runs after upstream regardless of outcome.
3. **C** — Dashboard task is the dedicated refresh type.
4. **A** — task-level `max_retries` handles transient failures.
5. **C** — table update trigger watches Delta commits.

</details>

## Summary

- **Lakeflow Jobs** (formerly Workflows / Jobs): a *job* is a DAG of *tasks*; a *run* is one execution.
- **Task types** the exam tests by name: **Notebook**, **SQL**, **Pipeline**, **Dashboard**, plus JAR / Python / dbt / Run-job / `for_each` / `if/else`.
- **DAG dependencies** via `depends_on` + `run_if`. `ALL_DONE` is the always-run-on-finish notification pattern.
- **Control flow**: `if/else` for branching, `for_each` for fan-out loops, **retries** (`max_retries`) for transient failures.
- **Triggers**: scheduled (time-based), file arrival, table update (data-driven), continuous, manual.
- **Time-based vs. data-driven**: time-based when source cadence is predictable; data-driven when it isn't or when you want to avoid empty runs.
- **Compute per task**: job cluster (default), serverless (fast start), all-purpose (avoid), SQL warehouse (for SQL/Dashboard).
- **Parameters and task values** carry state between tasks. Templates: `{{job.parameters.*}}`, `{{tasks.<key>.values.<key>}}`.
- **Monitoring**: run history for trends, task graph for upstream blockers, **repair run** for cheap re-execution of failed tasks only.
- **Production jobs run as a service principal**, not a human account.

Next up: **notebook 07 — CI/CD: Git Folders, Automation Bundles, Databricks CLI** — how to take all the notebooks, pipelines, and jobs we've built and promote them from dev to test to prod.